In [ ]:
# pip install presidio-analyzer presidio-anonymizer spacy azureml-fsspec pandas gliner2
# python -m spacy download en_core_web_lg

from pathlib import Path
import io
import time
import pandas as pd
from azureml.fsspec import AzureMachineLearningFileSystem
from redaction import PIIRedactor, GLiNER2Recognizer

In [ ]:
INPUT_URI = "azureml://subscriptions/<sub>/resourcegroups/<rg>/workspaces/<ws>/datastores/share_datastore/paths/yvonne-datastore/raw-call-transcripts/"

OUT_DIR = "<fill in output path here>"

MODEL_PATH = "fastino/gliner2-privacy-filter-PII-multi"
WINDOW = 2
SCORE_THRESHOLD = 0.5
MAX_FILES = 1   # set to None to run all files
TARGET_FILE = None  # e.g. "example_1.csv" to run one specific file, else None

In [ ]:
pii_redactor = PIIRedactor()

gliner2_recognizer = GLiNER2Recognizer(
    model_path=MODEL_PATH,
    threshold=SCORE_THRESHOLD,
)

pii_redactor.analyzer.registry.add_recognizer(gliner2_recognizer)
pii_redactor.analyzer.registry.remove_recognizer("SpacyRecognizer")

ENTITIES = PIIRedactor.PII_ENTITIES + [
    "PERSON",         # caught by GLiNER2 ("person" label); not in rule-based set
    "ACCOUNT_NUMBER", # caught by GLiNER2 and the 10-digit regex in PIIRedactor
]

In [ ]:
import re


def build_window(rows, i, window):
    """
    Combine row i with its neighbouring rows into a single string for analysis.

    GLiNER2 is a language model that needs surrounding context to correctly
    identify entities. For example, "6267" alone is ambiguous, but when seen
    next to "My contact is 9040..." it becomes clearly part of a phone number.
    By combining ±window rows, the model sees enough context to make that call.

    Steps:
      1. Determine the window slice: rows[max(0, i-window) : i+window+1]
      2. Concatenate the rows with '\n' separators.
      3. Track each row's [start, end] character range in the combined string.
      4. Return the combined string and the target row's character range
         (target_start, target_end) so callers can map detected spans back
         to row i later.

    Args:
        rows: full list of transcript rows
        i: index of the target row to analyse
        window: number of rows to include on each side

    Returns:
        combined (str): the joined window string
        target_start (int): char offset where row i begins in combined
        target_end (int): char offset where row i ends in combined
    """
    win_start = max(0, i - window)
    win_end = min(len(rows), i + window + 1)
    window_rows = rows[win_start:win_end]

    combined = ""
    row_ranges = []
    for j, text in enumerate(window_rows):
        start = len(combined)
        combined += text
        row_ranges.append((start, len(combined)))
        if j < len(window_rows) - 1:
            combined += "\n"

    target_start, target_end = row_ranges[i - win_start]
    return combined, target_start, target_end


def clamp_spans(results, target_start, target_end):
    """
    Filter and remap Presidio RecognizerResult spans to the target row.

    Presidio returns spans as character offsets in the combined window string.
    This function keeps only spans that fall (at least partially) within the
    target row's range, and converts their offsets to be relative to the
    start of the target row.

    Steps:
      1. For each RecognizerResult r in results:
         a. Clamp r.start to max(r.start, target_start) to ignore the
            portion that falls in preceding rows.
         b. Clamp r.end to min(r.end, target_end) to ignore the portion
            that falls in following rows.
         c. Subtract target_start to make offsets relative to the row.
         d. If the clamped span has positive length, keep it.
      2. Return spans sorted right-to-left (reverse=True) so apply_redactions
         can splice from the end without shifting earlier offsets.

    Args:
        results: list of presidio_analyzer.RecognizerResult from analyzer.analyze()
        target_start: start char offset of the target row in the combined string
        target_end: end char offset of the target row in the combined string

    Returns:
        list of (local_start, local_end, entity_type) tuples
    """
    spans = []
    for r in results:
        local_start = max(r.start, target_start) - target_start
        local_end = min(r.end, target_end) - target_start
        if local_start < local_end:
            spans.append((local_start, local_end, r.entity_type))
    return sorted(spans, key=lambda s: s[0], reverse=True)


def apply_redactions(text, spans):
    """
    Replace detected entity spans in text with <ENTITY_TYPE> tags.

    Spans must be sorted right-to-left (largest start first) so that each
    replacement does not shift the character offsets of spans still to be
    processed to the left.

    Steps:
      1. Iterate spans from rightmost to leftmost.
      2. For each (start, end, label), splice the tag into the text:
         text = text[:start] + '<LABEL>' + text[end:]
      3. Return the fully redacted string.

    Args:
        text (str): the target row's original text
        spans: list of (start, end, entity_type) sorted right-to-left

    Returns:
        str: text with all entity spans replaced by <ENTITY_TYPE> tags
    """
    for start, end, label in spans:
        text = text[:start] + f"<{label}>" + text[end:]
    return text


def fallback_spans(target_text, results, combined):
    """
    Recover spans for entities that clamp_spans missed due to cross-row detection.

    When an entity spans a row boundary (e.g. a phone number that starts in
    row i-1 and ends in row i), clamp_spans returns an empty span because the
    detected range straddles target_start. This fallback searches the target row
    directly for the entity's text value.

    Steps:
      1. For each RecognizerResult r, extract the detected value from combined.
      2. Exact match: if the value appears in target_text, record that position.
      3. Digit fallback: for cross-row entities (e.g. "9040\n6267"), extract all
         digit sequences from the value and find them in target_text. Redact from
         the first digit match to the last — this catches cases where only the
         tail digits of a number landed in the target row.
      4. Return spans sorted right-to-left for apply_redactions.

    Args:
        target_text (str): the target row's original text
        results: list of RecognizerResult from the combined window analysis
        combined (str): the combined window string (to extract entity values)

    Returns:
        list of (start, end, entity_type) tuples sorted right-to-left
    """
    spans = []
    for r in results:
        value = combined[r.start:r.end]
        if not value:
            continue

        if value in target_text:
            start = target_text.index(value)
            spans.append((start, start + len(value), r.entity_type))
            continue

        # Cross-row entity: extract digit sequences and find them in target row
        digit_seqs = list(dict.fromkeys(re.findall(r'\d+', value)))
        positions = []
        for seq in digit_seqs:
            for m in re.finditer(re.escape(seq), target_text):
                positions.append((m.start(), m.end()))

        if positions:
            start = min(p[0] for p in positions)
            end = max(p[1] for p in positions)
            spans.append((start, end, r.entity_type))

    return sorted(spans, key=lambda s: s[0], reverse=True)


def find_duplicate_spans(target_text, results, combined, covered_spans):
    """
    Ensure every occurrence of a detected entity value is redacted, not just the first.

    remove_overlapping_spans uses a greedy left-to-right algorithm. If a large span
    (e.g. SG_ADDRESS absorbing '8821 9234') sets last_end past a second occurrence
    of '8821 9234', that second occurrence gets silently dropped. This function does
    a second pass to catch any remaining uncovered occurrences.

    Steps:
      1. Build a set of already-covered (start, end) ranges from covered_spans.
      2. For each RecognizerResult r in results, extract the entity value from combined.
      3. Use re.finditer to locate ALL occurrences of that value in target_text.
      4. For each occurrence not already covered, add a new span with the same
         entity type and record it as covered to avoid double-adding.
      5. Return the list of additional spans to be merged with existing spans.

    Args:
        target_text (str): the target row's original text
        results: list of RecognizerResult from the combined window analysis
        combined (str): the combined window string (to extract entity values)
        covered_spans: existing spans already assigned to this row

    Returns:
        list of additional (start, end, entity_type) tuples
    """
    covered = [(s[0], s[1]) for s in covered_spans]
    extra = []
    seen = set()

    for r in results:
        value = combined[r.start:r.end].strip()
        if not value or value in seen:
            continue
        seen.add(value)

        for m in re.finditer(re.escape(value), target_text):
            already_covered = any(lo <= m.start() and m.end() <= hi for lo, hi in covered)
            if not already_covered:
                extra.append((m.start(), m.end(), r.entity_type))
                covered.append((m.start(), m.end()))

    return extra


def remove_overlapping_spans(spans):
    """
    Deduplicate overlapping spans using a greedy left-to-right sweep.

    When multiple entity types detect the same text region (e.g. SG_ADDRESS_BLOCK
    and SG_ADDRESS both covering 'Blk 45'), we keep the leftmost span and discard
    any subsequent span whose start falls before the previous span's end.

    Steps:
      1. Sort all spans by start position (ascending).
      2. Walk left to right, tracking last_end.
      3. Accept a span only if its start >= last_end (non-overlapping).
      4. Update last_end when a span is accepted.
      5. Re-sort accepted spans right-to-left for apply_redactions.

    Args:
        spans: list of (start, end, entity_type) tuples

    Returns:
        list of non-overlapping (start, end, entity_type) sorted right-to-left
    """
    sorted_asc = sorted(spans, key=lambda s: s[0])
    result = []
    last_end = -1
    for start, end, label in sorted_asc:
        if start >= last_end:
            result.append((start, end, label))
            last_end = end
    return sorted(result, key=lambda s: s[0], reverse=True)


# debug=True prints for every row:
#   - the raw text of the row
#   - results: every entity Presidio detected anywhere in the window
#   - spans: the final spans after clamp, fallback, duplicate expansion, and overlap removal
#
# To debug a specific file:
#   df = redact_csv(data, debug=True)
#
# To debug a small set of rows without loading a file:
#   rows = ["the account number is 9323284844", "just give me a moment", "ok", "9323284844"]
#   redact_rows(rows, debug=True)
def redact_rows(rows, debug=False):
    """
    Redact PII from a list of transcript rows using windowed analysis.

    Two-pass design:
      Pass 1 — Analysis:
        For every row i, build a window of ±WINDOW surrounding rows and run
        pii_redactor.analyzer.analyze() on the combined string. Both the
        rule-based recognizers (regex/spaCy) and GLiNER2 run together over
        the window, so GLiNER2 has enough context to correctly identify
        fragmented or ambiguous entities. Results are stored for all rows
        before any redaction begins.

      Pass 2 — Redaction:
        For each row i, convert the window-level results into row-level spans:
          1. clamp_spans: keep only spans that fall within row i's character
             range and remap offsets to be relative to that row.
          2. fallback_spans: if clamp yields nothing (entity crossed a row
             boundary), search the row directly for the entity value.
          3. Neighbour fallback: if own window found nothing, check the results
             of adjacent windows — a number mentioned in a nearby row might
             also appear in this row.
          4. find_duplicate_spans: scan for any remaining occurrences of
             detected values not yet covered by a span.
          5. remove_overlapping_spans: deduplicate any overlapping spans.
          6. apply_redactions: replace each span with <ENTITY_TYPE>.

    Args:
        rows: list of transcript row strings (the 'text' column values)
        debug (bool): if True, print detected results and final spans per row

    Returns:
        list of redacted strings, same length as rows
    """
    redacted = list(rows)

    # Pass 1: analyze each row's window and store results
    all_results = []
    for i in range(len(rows)):
        combined, target_start, target_end = build_window(rows, i, WINDOW)
        results = pii_redactor.analyzer.analyze(text=combined, entities=ENTITIES, language='en', score_threshold=SCORE_THRESHOLD)
        all_results.append((results, combined, target_start, target_end))

    # Pass 2: clamp spans to target row and apply redactions
    for i in range(len(rows)):
        results, combined, target_start, target_end = all_results[i]
        spans = clamp_spans(results, target_start, target_end)
        if not spans:
            spans = fallback_spans(rows[i], results, combined)
        if not spans:
            # Own window found nothing — check neighbouring windows
            for j in range(max(0, i - WINDOW), min(len(rows), i + WINDOW + 1)):
                if j == i:
                    continue
                nb_results, nb_combined, _, _ = all_results[j]
                spans = fallback_spans(rows[i], nb_results, nb_combined)
                if spans:
                    break
        extra = find_duplicate_spans(rows[i], results, combined, spans)
        spans = remove_overlapping_spans(spans + extra)
        if debug:
            print(f"\n--- row {i} ---")
            print(f"text:    {rows[i]!r}")
            print(f"results: {results}")
            print(f"spans:   {spans}")
        redacted[i] = apply_redactions(rows[i], spans)
    return redacted


def redact_csv(data, debug=False):
    """
    Read a CSV from bytes, redact the 'text' column, and return the DataFrame.

    Steps:
      1. Parse the CSV bytes into a DataFrame.
      2. Validate that a 'text' column exists.
      3. Extract rows as a list, filling NaN with empty strings.
      4. Run redact_rows() on the list.
      5. Write the redacted list back into df['text'] and return.

    Args:
        data (bytes): raw CSV file content
        debug (bool): passed through to redact_rows for per-row debug output

    Returns:
        pd.DataFrame with the 'text' column redacted
    """
    df = pd.read_csv(io.BytesIO(data))
    if "text" not in df.columns:
        raise ValueError("No 'text' column found in CSV")
    rows = df["text"].fillna("").tolist()
    df["text"] = redact_rows(rows, debug=debug)
    return df

In [ ]:
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

fs = AzureMachineLearningFileSystem(INPUT_URI)
folder = INPUT_URI.split("/paths/", 1)[1].rstrip("/")

if TARGET_FILE:
    csv_files = [TARGET_FILE]
else:
    csv_files = sorted(f for f in fs.ls("") if f.endswith(".csv"))
    if MAX_FILES:
        csv_files = csv_files[:MAX_FILES]

print(f"Found {len(csv_files)} CSV file(s)\n")

total_start = time.time()

for i, filepath in enumerate(csv_files, 1):
    filename = Path(filepath).name
    print(f"[{i}/{len(csv_files)}] {filename} ...", end=" ", flush=True)

    file_start = time.time()

    with fs.open(f"{folder}/{filename}", "rb") as f:
        data = f.read()

    df = redact_csv(data)
    df.to_csv(out_dir / filename, index=False)

    elapsed = time.time() - file_start
    print(f"done ({len(df)} rows, {elapsed:.1f}s, {elapsed/len(df):.2f}s/row)")

total_elapsed = time.time() - total_start
print(f"\nAll done in {total_elapsed:.1f}s. Redacted CSVs saved to: {out_dir}")